# Dedalus Pipeline Demo

## Project Goal

Dedalus is a quantum-enhanced join-order optimizer for SQL queries. It uses QUBO (Quadratic Unconstrained Binary Optimization) solvers, both simulated and hardware-based, to generate PostgreSQL optimizer hints. The pipeline then compares quantum-optimized execution against default PostgreSQL execution, measuring speedup and solution quality.

This notebook demonstrates the full Dedalus pipeline: query loading, analysis, QUBO-based solver optimization, default and hinted PostgreSQL execution, and runtime comparison.

**Target users:** Researchers evaluating quantum solvers for query optimization, developers benchmarking plan quality, and practitioners exploring quantum-inspired heuristics.

## Prerequisites

Before running this demo, ensure the required database containers are running:

- **Join Order Benchmark (JOB)** database: `docker compose up pg-job`
- **SampleDB** database: `docker compose up pg-sampledb`

## Pipeline Stages

1. **Load Query** &rarr; Reads SQL from file or inline, creates timestamped output directory
2. **Analyze Query** &rarr; Extracts join graph, cardinalities, and query plan metadata
3. **Build QUBO & Solve** &rarr; Constructs weight matrix from statistics, runs baseline DP + configured solver
4. **Execute PostgreSQL (Default)** &rarr; Runs query with default optimizer, records timing and plan
5. **Execute PostgreSQL (Hinted)** &rarr; Applies quantum-generated hint, compares execution time
6. **Update Cost Estimator** &rarr; Logs outcomes for online learning (if available)

In [1]:
import os
import sys

# Change to project root so relative paths resolve correctly
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)

# Import required modules
from app.config import PipelineConfig
from app.pipeline import DedalusPipeline

[Cost Estimator] PyTorch not available. Cost estimator will use fallback heuristics.


## Configuration / API

The user can provide a configuration dictionary to customize the pipeline. The `PipelineConfig` class in `app/config.py` handles parsing and validating these configurations, ensuring all required fields are present and values are valid before pipeline execution.

### Configuration Structure

The configuration is a nested dictionary with three main sections:
- **database**: Connection details for PostgreSQL
- **solver**: QUBO solver settings and execution options
- **query**: SQL query source and output settings

For valid values of each solver attribute, refer to `SOLVER_SCHEMA` in `app/config.py`, which defines allowed backends, solvers, and options.

**Key capabilities:**
- Test different query files from JOB, SampleDB databases
- Switch solver backends (D-Wave, Qiskit) and algorithms
- Enable pruning, benchmarking, or comparison modes

### Run Modes

The `run_mode` parameter controls how the pipeline executes:

- **`"normal"`** → Single execution run. Runs the solver once, generates a hint, executes both default and hinted PostgreSQL, records timings.
- **`"compare-pruning"`** → Benchmarks two QUBO formulations: `pruned` (search space reduced) vs `original` (full search space). Useful for evaluating pruning effectiveness.

### Solver Configuration

**D-Wave Backend:**
- `simulated_annealing` &rarr; Fast local simulation, good for prototyping and testing
- `exact_result` &rarr; Classical exact solver (brute force), for small problems and validation
- `Advantage_systemX.X` &rarr; Hardware QPUs, requires D-Wave API token
  - **Important setup**: Copy `configs/dwave/dwave-example.conf` to `configs/dwave/dwave.conf` and add your D-Wave API token.
- `hybrid_binary_quadratic_model_version2p`, `hybrid_constrained_quadratic_model_version1p` &rarr; Hybrid solvers (classical + quantum), accessible via D-Wave API

**Qiskit Backend:**
- `exact_result` &rarr; Exact statevector simulation
- `qaoa_sim`, `vqe_sim` &rarr; Quantum approximate algorithms on simulator

**Weight Modes:**
- `"cardinality"` &rarr; Default, uses join cardinality statistics
- `"cpu"` &rarr; CPU cost model estimation
- `"random"` &rarr; Random weights (for testing/comparison)

**Cardinality Types:**
- `"unfiltered"` &rarr; Raw cardinalities
- `"estimated"` &rarr; Refined estimates from query analyzer EXPLAIN

In [2]:
cfg = {
    "database": {
        "host": "pg-sampledb",                      # Database host (e.g., localhost, or Docker service name)
        "port": 5432,                               # Database port
        "user": "postgres",                         # Database username
        "password": "postgres",                     # Database password
        "name": "sampledb"                          # Database name
    },
    "solver": {
        "backend": "dwave",                         # Solver backend (dwave, qiskit, etc.)
        "solver": "simulated_annealing",            # Specific solver of selected backend
        "solver_opts": {"mode": "dwave"},           # Solver-specific options
        "weight_mode": "cardinality",               # Weight calculation mode for building QUBO weights
        "cardinality_type": "unfiltered",           # Cardinality type (relevant only when weight_mode = cardinality)
        "run_mode": "normal",                       # Execution mode
        "explain_format": "txt",                    # Query plan format (txt or json)
        "verbose": False,                           # Enable verbose logging intermediate results
        "export": True,                             # Export results to directory
        "normalize_weights": True,                  # Normalize weights
        "normalize_after_pruning": True             # Normalize weights before/after pruning 
    },
    "query": {
        "sql": None,                                # (Optional) Inline SQL query, alternative to file_path
        "file_path": "db/sampledb/queries/4rel/q3.sql",  # Path to query file
        "output_dir": "output"                      # Output directory for results
    }
}

In [3]:
# Configure the pipeline
config = PipelineConfig.from_dict(cfg)

## Execution

The pipeline will:

1. Analyze the query and extract join statistics.

2. Run baseline dynamic programming and the configured QUBO solver.

3. Generate a PostgreSQL optimizer hint from the solver output.

4. Execute the query with default PostgreSQL optimizer.

5. Execute the query with the quantum-enhanced hint.

6. Compare execution times and measure speedup.

In [4]:
# Run the Dedalus pipeline
pipeline = DedalusPipeline(config)
pipeline.run_pipeline()

[Cost Estimator] No trained model found, using heuristics

[Configuration]
{
  "database": {
    "user": "postgres",
    "password": "postgres",
    "host": "pg-sampledb",
    "port": 5432,
    "name": "sampledb"
  },
  "solver": {
    "backend": "dwave",
    "solver": "simulated_annealing",
    "solver_opts": {
      "mode": "dwave"
    },
    "weight_mode": "cardinality",
    "cardinality_type": "unfiltered",
    "normalize_weights": true,
    "normalize_after_pruning": true,
    "run_mode": "normal",
    "explain_format": "txt",
    "verbose": false,
    "export": true,
    "qubo_version": "pruned"
  },
  "query": {
    "sql": null,
    "file_path": "db/sampledb/queries/4rel/q3.sql",
    "output_dir": "output"
  }
}

[Query Analysis]
Query analysis completed.
=> File saved: output/query_20260520_164124/query_info_plan.json
=> File saved: output/query_20260520_164124/pipeline_run.json

[Solver]

Running Dynamic Programming (DP) Algorithm:
DP Elapsed: 0.080 ms
DP Tree: ['w', ['d', ['e

## Results Output

Pipeline results are saved to `output/<timestamp>/pipeline_run.json` if `export: True`. This includes:
- Query analysis and cardinalities
- Solver results (tree, cost, variables, execution time)
- Query execution plans (default and hinted)
- Runtime comparison and speedup ratio